# buffer-copy_-inplace — ex3: in-place EMA via mul_(1-m).add_(other, alpha=m) — zero-temp recipe

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `buffer-copy_-inplace`. Running the final beacon cell reports progress against the `PyTorch: in-place buffer copy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: in-place buffer copy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`buffer-copy_-inplace`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "buffer-copy_-inplace"
DD_SUBTOPIC = "PyTorch: in-place buffer copy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `copy_()` EMA vs in-place `mul_(...).add_(..., alpha=m)` recipe — same buffer, two routes

Ex1 wrote the EMA update through `running_mean.copy_(new_value)` where
`new_value = (1 - m) * running_mean + m * batch_mean` was built as a
fresh tensor first. The deepening move is to do the SAME math WITHOUT
ever allocating that intermediate — purely in-place:

```python
# Route A (ex1): allocate a temp, then copy_ overwrites the buffer.
running_mean.copy_((1 - m) * running_mean + m * batch_mean)

# Route B (ex3): two chained in-place ops, zero temps.
running_mean.mul_(1 - m).add_(batch_mean, alpha=m)
```

**Both preserve `data_ptr()`.** `copy_`, `mul_`, and `add_` all write
into the existing storage. So registered-buffer links survive either
route — `id(bn.running_mean)` is the same before and after.

**Route B allocates zero extra tensors.** Route A allocates two
intermediates (`(1-m)*running_mean` and `m*batch_mean`) plus a third for
the sum — all GC-able but allocator-churning in a tight loop.

**`add_(other, alpha=m)` is the standard idiom.** `tensor.add_(other,
alpha=k)` computes `tensor += k * other` in place without materializing
`k * other`. This is what `torch.optim.SGD.step()` uses internally for
the momentum buffer.

### Exercise 3 — in-place EMA via mul_(1-m).add_(other, alpha=m) — zero-temp recipe

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the chained `mul_(1-m).add_(other, alpha=m)` in-place EMA recipe to update a registered BatchNorm-style buffer without allocating any intermediate tensors, preserving the buffer's `data_ptr()` and `id()` identity.
> Keywords: mul_, add_, ema, in-place, zero-temp
> ```

**KCs targeted:** `in-place-mul-add-recipe`, `data-ptr-preserved-in-place`

Implement `ex3_ema_inplace(running_mean, batch_mean, momentum)`. Update `running_mean` IN PLACE to `(1 - momentum) * running_mean + momentum * batch_mean` using EXACTLY two chained in-place ops on `running_mean` — no temporary tensor allocation, no `.copy_(...)` call.

Required recipe:

```python
running_mean.mul_(1 - momentum).add_(batch_mean, alpha=momentum)
```

Inputs:
- `running_mean`: `Tensor`. The buffer to mutate.
- `batch_mean`: `Tensor` of the same shape as `running_mean`.
- `momentum`: `float` in `[0.0, 1.0]`.

Return value: `None`. The function mutates `running_mean` in place. The test asserts both `id(running_mean)` and `running_mean.data_ptr()` are preserved across the call — proof the registered-buffer link survives.

Constraints:
- DO NOT use `.copy_(...)` (ex1 already covered that route).
- DO NOT create a fresh tensor via `1 - m * running_mean + ...` and assign — that breaks the zero-temp invariant.
- DO NOT use `t.lerp_` either — the assertion is that you can hand-build the EMA from `mul_` and `add_`.

In [ ]:
def ex3_ema_inplace(running_mean: Tensor, batch_mean: Tensor, momentum: float) -> None:
    """Zero-temp in-place EMA: running_mean.mul_(1-m).add_(batch_mean, alpha=m)."""
    raise NotImplementedError()


def _test_ex3():
    def _test_ex3():
        # === Identity preservation: id() AND data_ptr() unchanged ===
        rm = t.tensor([1.0, 2.0, 3.0, 4.0])
        bm = t.tensor([5.0, 6.0, 7.0, 8.0])
        id_before = id(rm)
        ptr_before = rm.data_ptr()
        out = ex3_ema_inplace(rm, bm, 0.1)
        assert out is None, 'function must return None (mutates in place)'
        assert id(rm) == id_before, f'id changed — wrong route; expected {id_before}, got {id(rm)}'
        assert rm.data_ptr() == ptr_before, f'data_ptr changed — buffer reallocated; expected {ptr_before}, got {rm.data_ptr()}'

        # === Numeric correctness: (1-m)*rm + m*bm ===
        expected = 0.9 * t.tensor([1.0, 2.0, 3.0, 4.0]) + 0.1 * t.tensor([5.0, 6.0, 7.0, 8.0])
        assert t.allclose(rm, expected, atol=1e-7), f'numeric mismatch: got {rm}, expected {expected}'

        # === momentum=0 → buffer unchanged ===
        rm = t.tensor([1.0, 2.0])
        bm = t.tensor([99.0, 99.0])
        ex3_ema_inplace(rm, bm, 0.0)
        assert t.allclose(rm, t.tensor([1.0, 2.0])), f'momentum=0 should leave rm unchanged; got {rm}'

        # === momentum=1 → buffer becomes batch_mean ===
        rm = t.tensor([1.0, 2.0])
        bm = t.tensor([5.0, 6.0])
        ex3_ema_inplace(rm, bm, 1.0)
        assert t.allclose(rm, t.tensor([5.0, 6.0])), f'momentum=1 should make rm == bm; got {rm}'

        # === Works on a real registered buffer ===
        import torch.nn as nn
        class TinyBN(nn.Module):
            def __init__(self, n):
                super().__init__()
                self.register_buffer('running_mean', t.zeros(n))
        bn = TinyBN(3)
        original_id = id(bn.running_mean)
        original_ptr = bn.running_mean.data_ptr()
        ex3_ema_inplace(bn.running_mean, t.tensor([10.0, 20.0, 30.0]), 0.5)
        assert id(bn.running_mean) == original_id, 'registered buffer id broken'
        assert bn.running_mean.data_ptr() == original_ptr, 'registered buffer storage broken'
        assert t.allclose(bn.running_mean, t.tensor([5.0, 10.0, 15.0])), bn.running_mean

        # === Repeated calls — successive EMA steps converge toward batch_mean ===
        rm = t.zeros(2)
        bm = t.tensor([1.0, 1.0])
        ptr_initial = rm.data_ptr()
        for _ in range(100):
            ex3_ema_inplace(rm, bm, 0.1)
        assert rm.data_ptr() == ptr_initial, 'data_ptr changed across iterations'
        # 100 iters of EMA with m=0.1 toward [1, 1] should be very close.
        assert t.allclose(rm, t.ones(2), atol=1e-3), f'EMA failed to converge: {rm}'

        # === Shape: 2-D buffer (matches BatchNorm2d affine running stats) ===
        rm = t.zeros(3, 4)
        bm = t.ones(3, 4)
        ex3_ema_inplace(rm, bm, 0.25)
        assert t.allclose(rm, 0.25 * t.ones(3, 4)), rm
        print('ex3 ok')

    _test_ex3()
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_ema_inplace(running_mean, batch_mean, momentum):
    # Zero-temp in-place EMA. mul_ scales running_mean by (1-m) in place;
    # add_ with alpha=m adds m*batch_mean without materializing the product.
    running_mean.mul_(1 - momentum).add_(batch_mean, alpha=momentum)
```

**`add_(other, alpha=k)` avoids the multiply temp.** Without the `alpha=` kwarg you'd write `running_mean.add_(momentum * batch_mean)` — which allocates `momentum * batch_mean` first. The `alpha=` form does the multiply-and-add as a single fused kernel.

**Chain order matters.** `mul_(1-m).add_(other, alpha=m)` computes `((1-m) * rm) + m * other`. The reverse — `add_` first then `mul_` — would compute `(rm + other) * (1-m)`, a completely different expression.

**Why this isn't just style.** In a training loop with N modules and K BatchNorm layers each, the EMA update runs K*N times per step. Replacing three temps with zero per call adds up to measurable allocator savings on a hot path.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()